In [ ]:
import asyncio
import os
import subprocess
import sys
import pandas as pd
import re
import json
import random

# --- 1. УСТАНОВКА БИБЛИОТЕК ---
def install_libs():
    try:
        import playwright
        import bs4
    except ImportError:
        print("⏳ Устанавливаем библиотеки...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright", "beautifulsoup4", "pandas"])
        os.system("playwright install chromium")
        os.system("playwright install-deps")
        print("✅ Библиотеки готовы.")

install_libs()

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

# Business progress checkpoints for events-bot Kaggle status framework.
def _kaggle_business_progress(label, *, percent=None, phase=None, status="running", event="progress_checkpoint", **items):
    payload = {k: v for k, v in items.items() if v is not None}
    if label:
        payload["progress_label"] = str(label)
    if percent is not None:
        payload["progress_percent"] = int(max(0, min(100, round(float(percent)))))
    if phase:
        payload["phase"] = phase
    try:
        if "kaggle_status_update" in globals():
            kaggle_status_update(**payload)
        if "kaggle_status_event" in globals():
            kaggle_status_event(event, phase=phase, status=status, progress=payload)
    except Exception as exc:
        print(f"[kaggle_status] business progress failed: {exc}", flush=True)


async def stable_page_content(page, attempts=5):
    """Read the DOM after redirects/navigation have settled.

    Playwright rejects page.content() while a navigation is still replacing the
    main frame. Retry that transient boundary instead of aborting the remaining
    theatre sources and losing otherwise valid earlier outputs.
    """
    last_error = None
    for attempt in range(attempts):
        try:
            await page.wait_for_load_state('domcontentloaded', timeout=15000)
        except Exception:
            pass
        try:
            return await page.content()
        except Exception as exc:
            last_error = exc
            if attempt + 1 < attempts:
                await page.wait_for_timeout(1000)
    raise RuntimeError(f"Unable to read stable page content after {attempts} attempts: {last_error}") from last_error


# ==========================================
# ЧАСТЬ 1: ДРАМАТИЧЕСКИЙ ТЕАТР (DRAMTEATR)
# ==========================================

async def scrape_dram_schedule(page):
    url = "https://dramteatr39.ru/afisha"
    print(f"\n🎭 [Драмтеатр] Этап 1: Сканируем расписание на {url}...")

    try:
        await page.goto(url, timeout=90000, wait_until='domcontentloaded')
        await page.wait_for_timeout(3000)

        # Прокрутка
        scrolls = 6
        for _ in range(scrolls):
            await page.mouse.wheel(0, 4000)
            await page.wait_for_timeout(random.randint(1000, 3000))

        content = await stable_page_content(page)
        soup = BeautifulSoup(content, 'html.parser')

        events_list = []
        event_links = soup.find_all('a', href=re.compile(r'/spektakli/'))
        seen_identifiers = set()

        for link in event_links:
            href = link.get('href')
            full_link = f"https://dramteatr39.ru{href}" if href.startswith('/') else href

            title = link.get_text(strip=True)
            if len(title) < 2:
                continue

            container = link.find_parent(class_='affiche__container')
            card = container
            date_block = None
            if container:
                date_block = container.select_one('.affiche__date-block')
            else:
                # Fallback for unexpected markup changes.
                card = link.find_parent()
                if card:
                    date_block = card.select_one('.affiche__date-block')

            if not card:
                continue

            card_text = card.get_text(" ", strip=True).upper()

            # Дата
            date_text = "Дата не определена"
            day_text = ""
            month_text = ""
            time_text = ""

            if date_block:
                day_node = date_block.select_one('.date-list__day')
                month_node = date_block.select_one('.date-list__month')
                time_node = date_block.select_one('.date-list__time')

                if day_node:
                    day_text = day_node.get_text(strip=True)
                if month_node:
                    month_text = month_node.get_text(strip=True)
                if time_node:
                    spans = [s.get_text(strip=True) for s in time_node.find_all('span') if s.get_text(strip=True)]
                    if spans:
                        time_text = spans[-1]
                    else:
                        raw_time = time_node.get_text(" ", strip=True)
                        parts = [p for p in raw_time.split() if ":" in p]
                        if parts:
                            time_text = parts[-1]

            if day_text and month_text:
                date_text = f"{day_text} {month_text}"
                if time_text:
                    date_text = f"{date_text} {time_text}"

            # Статус
            status = "available"
            if "БИЛЕТОВ НЕТ" in card_text or "ПРОДАНО" in card_text:
                status = "sold_out"

            # Сцена
            if "ОСНОВНАЯ" in card_text:
                scene = "Основная"
            elif "СРЕДНЯЯ" in card_text:
                scene = "Средняя"
            else:
                scene = "Малая"

            event_id = f"{date_text}_{title}"
            if event_id in seen_identifiers:
                continue
            seen_identifiers.add(event_id)

            events_list.append({
                "title": title,
                "date_raw": date_text,
                "ticket_status": status,
                "scene": scene,
                "url": full_link,
                "location": "Драматический театр"
            })

        print(f"✅ [Драмтеатр] Найдено событий: {len(events_list)}")
        return events_list
    except Exception as e:
        print(f"❌ [Драмтеатр] Ошибка расписания: {e}")
        return []

async def scrape_dram_details(context, unique_urls):
    print(f"🕵️‍♂️ [Драмтеатр] Сбор деталей для {len(unique_urls)} спектаклей...")
    details_map = {}
    page = await context.new_page()

    for url in unique_urls:
        try:
            await page.goto(url, timeout=45000, wait_until='domcontentloaded')
            try:
                await page.wait_for_selector('h1', timeout=3000)
            except:
                pass

            content = await stable_page_content(page)
            soup = BeautifulSoup(content, 'html.parser')
            full_text = soup.get_text(" ", strip=True).upper()

            is_pushkin = "ПУШКИНСКАЯ КАРТА" in full_text

            images = []
            for a in soup.find_all('a', href=re.compile(r'\.(jpg|jpeg|png)$', re.IGNORECASE)):
                img_url = a['href']
                if img_url.startswith('/'):
                    img_url = f"https://dramteatr39.ru{img_url}"
                if img_url not in images:
                    images.append(img_url)

            if len(images) < 2:
                for img in soup.select('div.swiper-slide img, .gallery img, .content img'):
                    src = img.get('src')
                    if src and 'logo' not in src:
                        if src.startswith('/'):
                            src = f"https://dramteatr39.ru{src}"
                        if src not in images:
                            images.append(src)

            desc_text = ""
            desc_div = soup.find('div', class_=re.compile(r'description|text-block'))
            if not desc_div:
                header = soup.find(string=re.compile("О спектакле"))
                if header and header.parent:
                    parent = header.find_parent('div')
                    if parent:
                        desc_div = parent
            if desc_div:
                desc_text = desc_div.get_text("\n", strip=True)

            creators = ""
            c_div = soup.find('div', class_=re.compile(r'creators|team'))
            if c_div:
                creators = c_div.get_text(" | ", strip=True)

            details_map[url] = {
                "pushkin_card": is_pushkin,
                "photos": images,
                "description": desc_text[:1500],
                "creators": creators[:500]
            }
        except Exception:
            details_map[url] = {}

    await page.close()
    return details_map

async def run_dramteatr(browser):
    context = await browser.new_context(viewport={'width': 1920, 'height': 1080})
    page = await context.new_page()

    schedule = await scrape_dram_schedule(page)
    if not schedule:
        await context.close()
        return []

    unique_links = list(set(x['url'] for x in schedule))
    details = await scrape_dram_details(context, unique_links)

    await context.close()

    final_data = []
    for item in schedule:
        det = details.get(item['url'], {})
        final_data.append({**item, **det})

    return final_data

# ==========================================
# ЧАСТЬ 2: МУЗЫКАЛЬНЫЙ ТЕАТР (MUZTEATR)
# ==========================================

BASE_URL_MUZ = "https://muzteatr39.ru"

async def find_muz_afisha_link(page):
    try:
        await page.goto(BASE_URL_MUZ, timeout=30000, wait_until='domcontentloaded')
        link = page.get_by_role("link", name=re.compile("Афиша", re.IGNORECASE)).first
        href = await link.get_attribute("href")
        if href:
            return f"{BASE_URL_MUZ}{href}" if href.startswith("/") else href
    except:
        pass
    return f"{BASE_URL_MUZ}/afisha"

async def scrape_muz_schedule(page, afisha_url):
    print(f"\n🎻 [Музтеатр] Этап 1: Сканируем афишу: {afisha_url}...")
    try:
        await page.goto(afisha_url, timeout=60000, wait_until='domcontentloaded')
        await page.wait_for_timeout(2000)
    except:
        pass

    # Скролл
    try:
        for _ in range(3):
            await page.mouse.wheel(0, 4000)
            await page.wait_for_timeout(1000)
    except:
        pass

    content = await stable_page_content(page)
    soup = BeautifulSoup(content, 'html.parser')

    events_list = []
    unique_ids = set()
    MONTHS_MAP = {'ЯНВАР': 'ЯНВАРЯ', 'ФЕВРАЛ': 'ФЕВРАЛЯ', 'МАРТ': 'МАРТА', 'АПРЕЛ': 'АПРЕЛЯ', 'МАЙ': 'МАЯ', 'ИЮН': 'ИЮНЯ', 'ИЮЛ': 'ИЮЛЯ', 'АВГУСТ': 'АВГУСТА', 'СЕНТЯБР': 'СЕНТЯБРЯ', 'ОКТЯБР': 'ОКТЯБРЯ', 'НОЯБР': 'НОЯБРЯ', 'ДЕКАБР': 'ДЕКАБРЯ'}
    last_seen_month = "?"

    rows = soup.find_all('div', class_=re.compile(r'row\s+afisha'))
    for row in rows:
        date_col = row.find('div', class_=re.compile(r'afisha_data'))
        day_str, month_str, time_str = "?", last_seen_month, "00:00"

        if date_col:
            col_text = date_col.get_text(" ", strip=True).upper()
            day_div = date_col.find('div', class_='afisha_nom')
            if day_div:
                day_str = day_div.get_text(strip=True)
            else:
                day_match = re.search(r'^(\d{1,2})', col_text)
                if day_match:
                    day_str = day_match.group(1)

            for root, full_name in MONTHS_MAP.items():
                if root in col_text:
                    month_str = full_name
                    last_seen_month = full_name
                    break

            time_match = re.search(r'(\d{1,2}:\d{2})', col_text)
            if time_match:
                time_str = time_match.group(1).zfill(5)

        if not day_str.isdigit():
            continue
        date_final = f"{day_str} {month_str} {time_str}"

        title_div = row.find('div', class_='afisha_spek_title')
        if not title_div:
            continue

        title = ""
        link_tag = title_div.find('a')
        if link_tag:
            title = link_tag.get_text(strip=True)
            href = link_tag.get('href')
        else:
            badge = title_div.find('span', class_='badge')
            if badge:
                badge.decompose()
            title = title_div.get_text(strip=True)
            href = None

        if not href:
            continue
        full_link = f"{BASE_URL_MUZ}{href}" if href.startswith('/') else href

        badge = row.find('span', class_='badge')
        age = badge.get_text(strip=True) if badge else ""

        row_text_full = row.get_text(" ", strip=True).upper()
        status = "available"
        if "БИЛЕТОВ НЕТ" in row_text_full or "ПРОДАНО" in row_text_full:
            status = "sold_out"

        eid = f"{date_final}_{full_link}"
        if eid in unique_ids:
            continue
        unique_ids.add(eid)

        events_list.append({
            "title": title,
            "date_raw": date_final,
            "age_restriction": age,
            "ticket_status": status,
            "url": full_link,
            "location": "Музыкальный театр"
        })

    print(f"✅ [Музтеатр] Найдено событий: {len(events_list)}")
    return events_list

async def scrape_muz_details(context, unique_urls):
    print(f"🕵️‍♂️ [Музтеатр] Сбор деталей для {len(unique_urls)} событий...")
    details = {}
    page = await context.new_page()
    BAD_IMAGES = ['ulogin', 'provider', 'icon', 'logo', 'social', 'vk.com', 'ok.ru', 'blank', 'pixel']

    for url in unique_urls:
        try:
            if not url or 'http' not in url:
                continue
            try:
                await page.goto(url, timeout=30000, wait_until='domcontentloaded')
            except:
                pass

            content = await stable_page_content(page)
            soup = BeautifulSoup(content, 'html.parser')

            desc_text = ""
            meta_og = soup.find("meta", property="og:description")
            if meta_og and meta_og.get("content"):
                desc_text = meta_og["content"].strip()
            if not desc_text:
                meta_desc = soup.find("meta", attrs={"name": "description"})
                if meta_desc and meta_desc.get("content"):
                    desc_text = meta_desc["content"].strip()
            if not desc_text:
                content_div = soup.find('div', class_=re.compile(r'detail-text|news-detail|item-text'))
                if content_div:
                    raw = content_div.get_text("\n", strip=True)
                    lines = [l for l in raw.split('\n') if len(l) > 30 and 'войти' not in l.lower()]
                    desc_text = "\n".join(lines)[:2000]

            images = []
            for d in soup.find_all('div', style=re.compile(r'url\(')):
                match = re.search(r'url\([\'\"]?(.*?)[\'\"]?\)', d['style'])
                if match:
                    u = match.group(1)
                    if u.startswith('/'):
                        u = f"{BASE_URL_MUZ}{u}"
                    if not any(b in u.lower() for b in BAD_IMAGES):
                        images.append(u)

            container = soup.find('div', class_=re.compile(r'container|main|content')) or soup
            for img in container.find_all('img'):
                src = img.get('src')
                if src and not src.endswith('.svg') and not any(b in src.lower() for b in BAD_IMAGES):
                    if src.startswith('/'):
                        src = f"{BASE_URL_MUZ}{src}"
                    images.append(src)

            details[url] = {
                "pushkin_card": "ПУШКИНСКАЯ КАРТА" in soup.get_text().upper(),
                "description": desc_text,
                "photos": list(set(images))
            }
        except:
            details[url] = {}

    await page.close()
    return details

async def run_muzteatr(browser):
    context = await browser.new_context(viewport={'width': 1920, 'height': 1080})
    page = await context.new_page()

    afisha_url = await find_muz_afisha_link(page)
    events = await scrape_muz_schedule(page, afisha_url)

    if not events:
        await context.close()
        return []

    urls = list(set(x['url'] for x in events))
    details = await scrape_muz_details(context, urls)

    await context.close()

    final = []
    for ev in events:
        det = details.get(ev['url'], {})
        final.append({**ev, **det})
    return final

BASE_URL_SOBOR = "https://sobor39.ru"

# --- 2. ПАРСЕР СОБОРА ---
async def scrape_sobor_strict(page):
    target_url = "https://sobor39.ru/events/concerts/night/"
    print(f"⛪ Заходим на {target_url}...")

    try:
        await page.goto(target_url, timeout=90000, wait_until='domcontentloaded')
        await page.wait_for_timeout(5000)
    except Exception as e:
        print(f"⚠️ Предупреждение загрузки: {e}. Пробуем работать с тем, что есть.")

    # --- ПРОКРУТКА ---
    print("⬇️ Начинаем прокрутку для подгрузки событий...")

    for i in range(10):
        await page.mouse.wheel(0, 5000)
        await page.wait_for_timeout(3000)

        try:
            more_btns = page.locator("text=/Показать ещ[ёе]|Загрузить/i")
            if await more_btns.count() > 0:
                if await more_btns.first.is_visible():
                    await more_btns.first.click()
                    await page.wait_for_timeout(2000)
        except:
            pass

        if i % 2 == 0:
            print(f"   ...прокрутка {i+1}/10")

    content = await stable_page_content(page)
    soup = BeautifulSoup(content, 'html.parser')

    events_list = []
    unique_keys = set()

    items = soup.find_all('div', class_='list-item')
    print(f"🔍 Найдено блоков .list-item: {len(items)}")

    for item in items:
        try:
            # 1. ЗАГОЛОВОК
            title_tag = item.find('div', class_='list-t')
            if not title_tag:
                continue
            title = title_tag.get_text(strip=True)

            # 2. ДАТА (Раздельные классы)
            date_raw = "Unknown"
            day_node = item.find('div', class_='list-day')
            month_node = item.find('div', class_='list-month')
            time_node = item.find('div', class_='list-time')

            if day_node and month_node:
                d_txt = day_node.get_text(strip=True)
                m_txt = month_node.get_text(strip=True)
                t_txt = time_node.get_text(strip=True) if time_node else ""
                date_raw = f"{d_txt} {m_txt} {t_txt}".strip()

            # 3. ССЫЛКА И СТАТУС
            url = ""
            status = "available"

            other_block = item.find('div', class_='list-other')
            if other_block:
                btn = other_block.find('a', href=True)
                if btn:
                    link = btn['href']
                    url = link if link.startswith('http') else f"{BASE_URL_SOBOR}{link}"
                    btn_text = btn.get_text(strip=True).upper()
                    if "SOLD" in btn_text or "ПРОДАНО" in btn_text:
                        status = "sold_out"

            if not url:
                status = "unknown"

            # 4. КАРТИНКА
            photo_url = ""
            img_block = item.find('div', class_='list-img')
            if img_block:
                img = img_block.find('img')
                if img and img.get('src'):
                    src = img['src']
                    if 'svg' not in src:
                        photo_url = src if src.startswith('http') else f"{BASE_URL_SOBOR}{src}"

            # 5. ОПИСАНИЕ (Markdown)
            description = ""
            descr_node = item.find('div', class_='list-descr')
            if descr_node:
                # get_text(separator="\n") заменяет <br> и закрывающие теги на перенос строки
                description = descr_node.get_text(separator="\n", strip=True)

            # Дедубликация
            key = f"{date_raw}_{title}"
            if key in unique_keys:
                continue
            unique_keys.add(key)

            events_list.append({
                "title": title,
                "date_raw": date_raw,
                "ticket_status": status,
                "url": url,
                "photos": [photo_url] if photo_url else [],
                "description": description,
                "pushkin_card": False,
                "location": "Кафедральный собор"
            })
        except Exception as e:
            print(f"⚠️ Ошибка при разборе элемента: {e}")
            continue

    print(f"✅ Успешно обработано событий: {len(events_list)}")
    return events_list

async def run_sobor(browser):
    context = await browser.new_context(viewport={'width': 1920, 'height': 1080})
    page = await context.new_page()

    data = await scrape_sobor_strict(page)
    await context.close()
    return data
















# ==========================================
# ЧАСТЬ 3: ТРЕТЬЯКОВСКАЯ ГАЛЕРЕЯ
# ==========================================

BASE_URL_TRETYAKOV = "https://kaliningrad.tretyakovgallery.ru"
MAX_EVENTS_TO_PROCESS = 1000
MAX_ARROW_CLICKS = 10

MONTHS_RU = {
    "января": 1, "февраля": 2, "марта": 3, "апреля": 4, "мая": 5, "июня": 6,
    "июля": 7, "августа": 8, "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12
}


def deduplicate_tretyakov_events(events):
    """
    Дедупликация: при совпадении (дата, время, зал) оставляем
    событие исполнителя (direct_url_date), удаляем фестиваль (all_dates_extracted).
    Фото из удаляемых событий добавляются к оставленному.
    """
    groups = {}
    for e in events:
        key = (e.get('parsed_date'), e.get('parsed_time'), e.get('location'))
        if key not in groups:
            groups[key] = []
        groups[key].append(e)
    
    result = []
    duplicates_removed = 0
    
    for key, group in groups.items():
        if len(group) == 1:
            result.append(group[0])
        else:
            direct = [e for e in group if e.get('source_type') == 'direct_url_date']
            other = [e for e in group if e.get('source_type') != 'direct_url_date']
            
            if direct:
                kept = direct[0].copy()
                # Merge photos from removed events
                all_photos = list(kept.get('photos', []))
                for removed in other:
                    for photo in removed.get('photos', []):
                        if photo and photo not in all_photos:
                            all_photos.append(photo)
                kept['photos'] = all_photos
                result.append(kept)
                duplicates_removed += len(group) - 1
            else:
                # No direct_url_date winner: keep records with different titles,
                # drop only exact title duplicates in the same date/time/location bucket.
                seen_titles = set()
                kept_count = 0
                for item in group:
                    norm_title = re.sub(r"\s+", " ", (item.get('title') or '').strip().lower())
                    key_title = norm_title or f"__empty__{kept_count}"
                    if key_title in seen_titles:
                        continue
                    seen_titles.add(key_title)
                    result.append(item)
                    kept_count += 1
                duplicates_removed += max(0, len(group) - kept_count)
    
    if duplicates_removed > 0:
        print(f"   🔄 Дедупликация: удалено {duplicates_removed} дубликатов")
    
    return result


async def scrape_tretyakov_events_list(page):
    """Scrape events from /events/ page, extracting detail_url and ticket_url."""
    url = f"{BASE_URL_TRETYAKOV}/events/"
    print(f"\n🖼️ [Третьяковка] Scanning: {url}")
    
    await page.goto(url, timeout=60000, wait_until='domcontentloaded')
    await page.wait_for_timeout(3000)
    
    for _ in range(3):
        await page.mouse.wheel(0, 3000)
        await page.wait_for_timeout(random.randint(1000, 1500))
    
    events = await page.evaluate("""
        () => {
            const events = [];
            const seen = new Set();
            const BASE = 'https://kaliningrad.tretyakovgallery.ru';
            
            document.querySelectorAll('.card').forEach(card => {
                const titleEl = card.querySelector('.card_title');
                if (!titleEl) return;
                
                const title = titleEl.innerText.trim();
                if (title.toUpperCase().includes('ЭКСКУРСИЯ')) return;
                
                // Get detail_url from onclick
                let detailUrl = null;
                const onclick = card.getAttribute('onclick');
                if (onclick) {
                    const match = onclick.match(/window\\.open\\(['\"]([^'\"]+)['\"]/);
                    if (match) detailUrl = match[1];
                }
                
                // Get ticket_url
                let ticketUrl = null;
                const ticketLink = card.querySelector('a[href*="tickets"]');
                if (ticketLink) {
                    let href = ticketLink.getAttribute('href');
                    if (href.startsWith('//')) ticketUrl = 'https:' + href;
                    else ticketUrl = href;
                }
                
                if (ticketUrl && ticketUrl.includes('timepad')) return;
                
                // Get photo
                let photo = null;
                const img = card.querySelector('img.card_img');
                if (img && img.src) {
                    photo = img.src.startsWith('/') ? BASE + img.src : img.src;
                }
                
                // Get location
                const text = card.innerText.toUpperCase();
                let location = 'Третьяковка Калининград';
                if (text.includes('АТРИУМ')) location = 'Атриум';
                else if (text.includes('КИНОЗАЛ')) location = 'Кинозал';
                
                const key = title + ticketUrl;
                if (seen.has(key)) return;
                seen.add(key);
                
                if (ticketUrl) {
                    events.push({
                        title_raw: title,
                        detail_url: detailUrl,
                        ticket_url: ticketUrl,
                        photo: photo,
                        location: location
                    });
                }
            });
            return events;
        }
    """)
    
    print(f"   ✅ Found {len(events)} events")
    return events[:MAX_EVENTS_TO_PROCESS]


async def scrape_tretyakov_detail(page, detail_url):
    """Visit detail page for title, FULL description, date/time, AND direct_ticket_url.
    
    CRITICAL: For Pianissimo performers, detail page has button with direct URL
    containing the correct date (e.g. /2026-01-30/20:00:00). This prevents phantom events.
    """
    import re
    import datetime
    
    if not detail_url:
        return {"title": None, "description": None, "parsed_date": None, "parsed_time": None, "direct_ticket_url": None}
    
    full_url = f"{BASE_URL_TRETYAKOV}{detail_url}" if detail_url.startswith('/') else detail_url
    print(f"   📄 Detail: {full_url}")
    
    try:
        await page.goto(full_url, timeout=30000, wait_until='domcontentloaded')
        await page.wait_for_timeout(2000)
        
        # Title
        title = None
        h1 = await page.query_selector('h1')
        if h1:
            title = (await h1.inner_text()).strip()
        
        # FULL description - collect ALL paragraphs
        description_parts = []
        paragraphs = await page.query_selector_all('p')
        for p in paragraphs:
            text = (await p.inner_text()).strip()
            if len(text) < 30:
                continue
            if any(skip in text.lower() for skip in ['cookie', 'политик', 'hours', 'работаем']):
                continue
            description_parts.append(text)
        description = '\n\n'.join(description_parts) if description_parts else None
        
        # Date and time from page text
        body_text = await page.inner_text("body")
        parsed_date = None
        parsed_time = None
        today = datetime.date.today()
        
        for match in re.finditer(r'(\d{1,2})\s+([а-яё]+)\s*,?\s*(?:в|В)\s*(\d{1,2}:\d{2})', body_text, re.IGNORECASE):
            day = int(match.group(1))
            month_name = match.group(2).lower().strip('.,')
            time_str = match.group(3)
            
            month_num = MONTHS_RU.get(month_name)
            if not month_num:
                continue
            
            year = today.year + (1 if today.month >= 10 and month_num < 3 else 0)
            
            try:
                date_obj = datetime.date(year, month_num, day)
                if date_obj >= today:
                    parsed_date = date_obj.isoformat()
                    parsed_time = time_str
                    break
            except:
                continue
        
        # CRITICAL: Extract direct ticket URL from "Buy ticket" button
        # Format: /tickets/#/buy/event/42168/2026-01-30/20:00:00
        direct_ticket_url = None
        ticket_links = await page.query_selector_all('a[href*="tickets"]')
        for tl in ticket_links:
            href = await tl.get_attribute('href')
            if href and '/buy/event/' in href and re.search(r'/\d{4}-\d{2}-\d{2}/', href):
                direct_ticket_url = href
                print(f"      🎫 Direct URL: {href[:60]}...")
                break
        
        if parsed_date and parsed_time:
            print(f"      📅 Detail date: {parsed_date} {parsed_time}")
        
        return {
            "title": title,
            "description": description,
            "parsed_date": parsed_date,
            "parsed_time": parsed_time,
            "direct_ticket_url": direct_ticket_url
        }
    except Exception as e:
        print(f"      ⚠️ Detail error: {e}")
        return {"title": None, "description": None, "parsed_date": None, "parsed_time": None, "direct_ticket_url": None}


async def get_prices_from_all_sectors(page):
    """Extract min and max prices by clicking each sector."""
    import re
    
    all_prices = []
    
    sector_labels = await page.query_selector_all('label.select-sector-button')
    if sector_labels:
        for sector in sector_labels:
            try:
                await sector.click()
                await page.wait_for_timeout(1000)
                
                price_el = await page.query_selector('.ticket-price')
                if price_el:
                    price_text = await price_el.inner_text()
                    match = re.search(r'(\d+)', price_text)
                    if match:
                        all_prices.append(int(match.group(1)))
            except:
                pass
    
    if not all_prices:
        # Fallback: search for any price
        price_el = await page.query_selector('.ticket-price')
        if price_el:
            price_text = await price_el.inner_text()
            match = re.search(r'(\d+)', price_text)
            if match:
                all_prices.append(int(match.group(1)))
    
    if all_prices:
        return min(all_prices), max(all_prices)
    return None, None


async def scrape_tretyakov_tickets_all_dates(page, ticket_url):
    """Parse ticket page for ALL dates using calendar navigation."""
    import datetime
    
    full_url = f"{BASE_URL_TRETYAKOV}{ticket_url}" if ticket_url.startswith('/') else ticket_url
    print(f"   🎫 Tickets: {full_url[:60]}...")
    today = datetime.date.today()
    results = []
    
    try:
        try:
            await page.goto(full_url, timeout=60000, wait_until='networkidle')
        except Exception:
            await page.goto(full_url, timeout=60000, wait_until='domcontentloaded')
        await page.wait_for_timeout(3000)
        
        # Collect ALL active dates by navigating calendar
        all_dates = set()
        for click in range(MAX_ARROW_CLICKS):
            visible = await page.evaluate("""() => {
                const items = [];
                document.querySelectorAll('div.item.active').forEach(item => {
                    const dayEl = item.querySelector('.calendarDay');
                    const monthEl = item.querySelector('.calendarMonth');
                    if (dayEl) items.push({ day: dayEl.innerText.trim(), month: monthEl ? monthEl.innerText.trim().toLowerCase() : '' });
                });
                return items;
            }""")
            
            for d in visible:
                all_dates.add((d['day'], d['month']))
            
            # Click right arrow
            # Click right arrow to see more dates
            arrow = await page.query_selector('.week-calendar-arrow.week-calendar-next')
            if arrow:
                is_visible = await arrow.is_visible()
                if is_visible:
                    await arrow.click()
                    await page.wait_for_timeout(800)
                else:
                    break
            else:
                break
        
        print(f"      📅 Active dates: {len(all_dates)}")
        
        # Reload to start fresh
        try:
            await page.goto(full_url, timeout=60000, wait_until='networkidle')
        except Exception:
            await page.goto(full_url, timeout=60000, wait_until='domcontentloaded')
        await page.wait_for_timeout(2000)
        
        for (day_str, month_str) in sorted(all_dates, key=lambda x: (MONTHS_RU.get(x[1], 0), int(x[0]))):
            month_num = MONTHS_RU.get(month_str, 1)
            year = today.year + (1 if today.month >= 10 and month_num < 3 else 0)
            
            try:
                date_obj = datetime.date(year, month_num, int(day_str))
                if date_obj < today:
                    continue
            except:
                continue
            
            date_iso = date_obj.isoformat()
            date_raw = f"{day_str} {month_str}"
            
            # Navigate calendar to find this date (it may be on another page)
            date_found = False
            for nav_attempt in range(MAX_ARROW_CLICKS):
                clicked = await page.evaluate(f"""() => {{ 
                    let found = false;
                    document.querySelectorAll('div.item.active').forEach(i => {{ 
                        const dayEl = i.querySelector('.calendarDay'); 
                        if (dayEl && dayEl.innerText.trim() === '{day_str}') {{
                            i.click();
                            found = true;
                        }}
                    }}); 
                    return found;
                }}""")
                
                if clicked:
                    date_found = True
                    await page.wait_for_timeout(1500)
                    break
                
                # Click right arrow to navigate
                arrow = await page.query_selector('.week-calendar-arrow.week-calendar-next')
                if arrow and await arrow.is_visible():
                    await arrow.click()
                    await page.wait_for_timeout(600)
                else:
                    break
            
            if not date_found:
                print(f"         ⚠️ Date {day_str} not found in calendar")
                continue
            
            # Get times
            times = await page.evaluate("""() => [...document.querySelectorAll('label.select-time-button:not(.disabled)')].map(b => b.innerText.trim().match(/^\\d{1,2}:\\d{2}$/)?.[0]).filter(Boolean)""")
            
            if not times:
                times = ['00:00']
            
            for time_str in times:
                # Click time
                await page.evaluate(f"""() => {{ 
                    document.querySelectorAll('label.select-time-button').forEach(b => {{ 
                        if (b.innerText.includes('{time_str}')) b.click(); 
                    }}); 
                }}""")
                await page.wait_for_timeout(1000)
                
                # Get min/max prices from all sectors
                price_min, price_max = await get_prices_from_all_sectors(page)
                
                body = await page.inner_text("body")
                if "все билеты проданы" in body.lower():
                    status = "sold_out"
                elif price_min:
                    status = "available"
                else:
                    status = "unknown"
                
                results.append({
                    "parsed_date": date_iso,
                    "parsed_time": time_str,
                    "date_raw": f"{date_raw} в {time_str}",
                    "ticket_price_min": price_min,
                    "ticket_price_max": price_max,
                    "ticket_status": status,
                })
        
        return results
    except Exception as e:
        print(f"      ⚠️ Ticket error: {e}")
        return []


async def run_tretyakov(browser):
    """Main parser with all improvements."""
    import re
    
    context = await browser.new_context(viewport={'width': 1920, 'height': 1080})
    await context.route("**/*{google,yandex,metrika,analytics}*", lambda route: route.abort())
    
    list_page = await context.new_page()
    detail_page = await context.new_page()
    ticket_page = await context.new_page()

    events_raw = await scrape_tretyakov_events_list(list_page)
    if not events_raw:
        await context.close()
        return []

    all_events = []
    
    for idx, event in enumerate(events_raw):
        print(f"\n📌 [{idx+1}/{len(events_raw)}] {event['title_raw'][:50]}...")
        
        # Clean ticket_url
        raw_url = event['ticket_url']
        if raw_url.startswith(BASE_URL_TRETYAKOV):
            raw_url = raw_url[len(BASE_URL_TRETYAKOV):]
        clean_url = re.sub(r'/\d{4}-\d{2}-\d{2}/\d{2}:\d{2}(:\d{2})?$', '', raw_url)
        
        # Get detail info
        detail = await scrape_tretyakov_detail(detail_page, event.get('detail_url'))
        title = detail['title'] or event['title_raw']
        description = detail['description']
        detail_date = detail.get('parsed_date')
        detail_time = detail.get('parsed_time')
        direct_ticket_url = detail.get('direct_ticket_url')
        
        photo = event['photo']
        if photo and photo.startswith('/'):
            photo = f"{BASE_URL_TRETYAKOV}{photo}"
        
        # CASE 1: Direct URL exists (Pianissimo performer)
        if direct_ticket_url:
            url_match = re.search(r'/(\d{4}-\d{2}-\d{2})/(\d{2}:\d{2})', direct_ticket_url)
            if url_match:
                specific_date = url_match.group(1)
                specific_time = url_match.group(2)
                print(f"      🎯 Using direct URL date: {specific_date} {specific_time}")
                
                # Get price from this specific date
                await ticket_page.goto(direct_ticket_url, timeout=60000, wait_until='networkidle')
                await ticket_page.wait_for_timeout(2000)
                price_min, price_max = await get_prices_from_all_sectors(ticket_page)
                
                body = await ticket_page.inner_text("body")
                status = "sold_out" if "все билеты проданы" in body.lower() else ("available" if price_min else "unknown")
                
                # Format date_raw
                day = int(specific_date.split('-')[2])
                month_num = int(specific_date.split('-')[1])
                month_names = {1: 'января', 2: 'февраля', 3: 'марта', 4: 'апреля', 5: 'мая', 6: 'июня',
                              7: 'июля', 8: 'августа', 9: 'сентября', 10: 'октября', 11: 'ноября', 12: 'декабря'}
                date_raw = f"{day} {month_names.get(month_num, '')} в {specific_time}"
                
                all_events.append({
                    "title": title,
                    "description": description,
                    "date_raw": date_raw,
                    "parsed_date": specific_date,
                    "parsed_time": specific_time,
                    "ticket_status": status,
                    "ticket_price_min": price_min,
                    "ticket_price_max": price_max,
                    "url": direct_ticket_url,
                    "photos": [photo] if photo else [],
                    "location": event['location'],
                    "scene": event['location'] if event['location'] in ["Атриум", "Кинозал"] else "",
                    "source_type": "direct_url_date"
                })
                continue
        
        # CASE 2: No direct URL - get ALL dates from calendar
        entries = await scrape_tretyakov_tickets_all_dates(ticket_page, clean_url)

        # Some ticket widgets may expose incorrect/shifted calendar date while
        # detail page contains the authoritative schedule date/time.
        if entries and detail_date:
            has_detail_date = any(e.get('parsed_date') == detail_date for e in entries)
            if not has_detail_date:
                month_names = {1: 'января', 2: 'февраля', 3: 'марта', 4: 'апреля', 5: 'мая', 6: 'июня',
                              7: 'июля', 8: 'августа', 9: 'сентября', 10: 'октября', 11: 'ноября', 12: 'декабря'}
                detail_time_safe = detail_time or entries[0].get('parsed_time') or '00:00'
                try:
                    y, m, d = [int(x) for x in str(detail_date).split('-')]
                    detail_raw = f"{d} {month_names.get(m, '')} в {detail_time_safe}"
                except Exception:
                    detail_raw = f"{detail_date} {detail_time_safe}".strip()
                entries = [{
                    "parsed_date": detail_date,
                    "parsed_time": detail_time_safe,
                    "date_raw": detail_raw,
                    "ticket_price_min": entries[0].get('ticket_price_min'),
                    "ticket_price_max": entries[0].get('ticket_price_max'),
                    "ticket_status": entries[0].get('ticket_status', 'unknown'),
                }]

        if entries:
            for e in entries:
                base = clean_url if not clean_url.startswith('/') else f"{BASE_URL_TRETYAKOV}{clean_url}"
                direct_url = f"{base}/{e['parsed_date']}/{e['parsed_time']}:00"
                
                all_events.append({
                    "title": title,
                    "description": description,
                    "date_raw": e['date_raw'],
                    "parsed_date": e['parsed_date'],
                    "parsed_time": e['parsed_time'],
                    "ticket_status": e['ticket_status'],
                    "ticket_price_min": e['ticket_price_min'],
                    "ticket_price_max": e['ticket_price_max'],
                    "url": direct_url,
                    "photos": [photo] if photo else [],
                    "location": event['location'],
                    "scene": event['location'] if event['location'] in ["Атриум", "Кинозал"] else "",
                    "source_type": "all_dates_extracted"
                })
        else:
            # Fallback: no calendar dates found. Preserve detail-page date/time if available.
            fallback_url = clean_url if not clean_url.startswith('/') else f"{BASE_URL_TRETYAKOV}{clean_url}"
            fallback_date = detail_date
            fallback_time = detail_time
            fallback_date_raw = ""
            if fallback_date:
                month_names = {1: 'января', 2: 'февраля', 3: 'марта', 4: 'апреля', 5: 'мая', 6: 'июня',
                              7: 'июля', 8: 'августа', 9: 'сентября', 10: 'октября', 11: 'ноября', 12: 'декабря'}
                try:
                    y, m, d = [int(x) for x in str(fallback_date).split('-')]
                    month_text = month_names.get(m, '')
                    if fallback_time:
                        fallback_date_raw = f"{d} {month_text} в {fallback_time}"
                    else:
                        fallback_date_raw = f"{d} {month_text}"
                except Exception:
                    fallback_date_raw = str(fallback_date)
            all_events.append({
                "title": title,
                "description": description,
                "date_raw": fallback_date_raw,
                "parsed_date": fallback_date,
                "parsed_time": fallback_time,
                "ticket_status": "unknown",
                "ticket_price_min": None,
                "ticket_price_max": None,
                "url": fallback_url,
                "photos": [photo] if photo else [],
                "location": event['location'],
                "scene": event['location'] if event['location'] in ["Атриум", "Кинозал"] else "",
                "source_type": "detail_fallback" if fallback_date else "no_dates"
            })

    # Deduplicate
    all_events = deduplicate_tretyakov_events(all_events)
    
    print(f"\n🎉 [Третьяковка] Total: {len(all_events)} events (after dedup)")
    await context.close()
    return all_events


# --- ЗАПУСК ---
async def main():
    config = {}
    target = "all"
    if os.path.exists('run_config.json'):
        config = json.load(open('run_config.json'))
        target = config.get('target_source', 'all')

    print("🚀 ЗАПУСК СБОРА ДАННЫХ...")
    print("ℹ️ [INFO] Запускаем Драмтеатр, Музтеатр, Кафедральный собор и Третьяковскую галерею.")
    active_sources = [name for name in ("dramteatr", "muzteatr", "sobor", "tretyakov") if target in ("all", name)]
    _kaggle_business_progress(f"источники 0/{len(active_sources)}", percent=10, phase="parse", sources_total=len(active_sources), sources_done=0)

    results = {}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        data_dram = []
        data_muz = []
        data_sobor = []
        data_tretyakov = []

        if target in ("all", "dramteatr"):
            data_dram = await run_dramteatr(browser)
            results["dramteatr"] = data_dram
            _kaggle_business_progress("dramteatr готов · события " + str(len(data_dram)), percent=30, phase="parse", source="dramteatr", events_parsed=len(data_dram), sources_done=sum(1 for key in results if key != "all"), sources_total=len(active_sources))
            if data_dram:
                with open('dramteatr.json', 'w', encoding='utf-8') as f:
                    json.dump(data_dram, f, ensure_ascii=False, indent=4)
                pd.DataFrame(data_dram).to_csv('dramteatr.csv', index=False)
                print(f"💾 Dramteatr saved: {len(data_dram)}")

        if target in ("all", "muzteatr"):
            data_muz = await run_muzteatr(browser)
            results["muzteatr"] = data_muz
            _kaggle_business_progress("muzteatr готов · события " + str(len(data_muz)), percent=50, phase="parse", source="muzteatr", events_parsed=len(data_muz), sources_done=sum(1 for key in results if key != "all"), sources_total=len(active_sources))
            if data_muz:
                with open('muzteatr.json', 'w', encoding='utf-8') as f:
                    json.dump(data_muz, f, ensure_ascii=False, indent=4)
                df_muz = pd.DataFrame(data_muz)
                if 'photos' in df_muz.columns:
                    df_muz['photos'] = df_muz['photos'].apply(lambda x: ' | '.join(x) if isinstance(x, list) else x)
                df_muz.to_csv('muzteatr.csv', index=False)
                print(f"💾 Muzteatr saved: {len(data_muz)}")

        if target in ("all", "sobor"):
            data_sobor = await run_sobor(browser)
            results["sobor"] = data_sobor
            _kaggle_business_progress("sobor готов · события " + str(len(data_sobor)), percent=70, phase="parse", source="sobor", events_parsed=len(data_sobor), sources_done=sum(1 for key in results if key != "all"), sources_total=len(active_sources))

        if target in ("all", "tretyakov"):
            data_tretyakov = await run_tretyakov(browser)
            results["tretyakov"] = data_tretyakov
            _kaggle_business_progress("tretyakov готов · события " + str(len(data_tretyakov)), percent=90, phase="parse", source="tretyakov", events_parsed=len(data_tretyakov), sources_done=sum(1 for key in results if key != "all"), sources_total=len(active_sources))

        await browser.close()

    all_data = []
    for chunk in results.values():
        if chunk:
            all_data.extend(chunk)

    if all_data:
        results["all"] = all_data
    _kaggle_business_progress(f"готово · события {len(all_data)}", percent=100, phase="report", events_parsed=len(all_data))

    return results

result = await main()

sobor_data = result.get("sobor", []) if result else []
tretyakov_data = result.get("tretyakov", []) if result else []
all_data = result.get("all", []) if result else []

if sobor_data:
    with open('sobor.json', 'w', encoding='utf-8') as f:
        json.dump(sobor_data, f, ensure_ascii=False, indent=4)

    df = pd.DataFrame(sobor_data)
    if 'photos' in df.columns:
        df['photo_preview'] = df['photos'].apply(lambda x: x[0] if x else "")

    # Обрезаем описание для превью в консоли
    if 'description' in df.columns:
        df['desc_preview'] = df['description'].apply(lambda x: (x[:50] + '...') if len(x) > 50 else x)

    df.drop(columns=['photos', 'desc_preview'], inplace=True, errors='ignore')
    df.to_csv('sobor.csv', index=False)

if tretyakov_data:
    with open('tretyakov.json', 'w', encoding='utf-8') as f:
        json.dump(tretyakov_data, f, ensure_ascii=False, indent=4)

    df_tretyakov = pd.DataFrame(tretyakov_data)
    if 'photos' in df_tretyakov.columns:
        df_tretyakov['photos'] = df_tretyakov['photos'].apply(lambda x: ' | '.join(x) if isinstance(x, list) else x)
    df_tretyakov.to_csv('tretyakov.csv', index=False)

if all_data:
    df_all = pd.DataFrame(all_data)
    if 'photos' in df_all.columns:
        df_all['photo_preview'] = df_all['photos'].apply(lambda x: x[0] if isinstance(x, list) and x else "")
    if 'description' in df_all.columns:
        df_all['desc_preview'] = df_all['description'].apply(lambda x: (x[:50] + '...') if len(x) > 50 else x)
    else:
        df_all['desc_preview'] = ""

    print(f"\n🎉 ИТОГ: {len(all_data)} событий.")
    pd.set_option('display.max_colwidth', 40)
    print(df_all[['date_raw', 'title', 'desc_preview']].head(10))
else:
    print("Данные не найдены.")
